# 03 · Comparar modelos de regresión

**Clases 6, 7, 10, 11 y 12:** árbol de regresión, Random Forest, XGBoost, variables derivadas, codificación, validación y métricas.

**Pregunta:** ¿podemos estimar la potencia de una planta a partir de irradiancia, temperatura, viento y fecha?

Se comparan regresión lineal simple, múltiple, árbol, Random Forest y XGBoost. Target: potencia / capacidad nominal (MW/MW). Todos aprenden de mediciones reales.

**Particiones China:** train hasta marzo de 2019; selección en abril; prueba mayo–junio. La selección usa el menor RMSE de validación; si RF está a menos del 1% del mínimo, se prefiere RF. Los meses ya se analizaron en trabajos previos: es evaluación retrospectiva, no una nueva prueba ciega.

La máscara de entrenamiento elimina apagones para aprender producción sana. La evaluación conserva los apagones y excluye solamente registros inválidos. Los datos y reglas completas están en `src/academico.py`.

In [ ]:
from pathlib import Path
import os, sys
if 'google.colab' in sys.modules:
    import subprocess
    root = Path('/content/tpo-solar')
    if not root.exists():
        subprocess.run(['git','clone','https://github.com/imelinc/tpo-solar.git',str(root)],check=True)
    os.chdir(root)
    subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'],check=True)
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.config import DATA_RAW, DATA_PROCESSED
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go


In [ ]:
from src.academico import construir
# Ejecuta realmente los cinco modelos, la selección y los escenarios de evaluación.
# Guarda la evidencia consumida por el dashboard, sin escribir métricas manualmente.
evidencia = construir()


In [ ]:
R=evidencia['regresion']
filas=[{'modelo':r['nombre'], 'particion':p, **r[p]} for r in R['comparacion'] for p in ['train','validacion','test']]
comparacion=pd.DataFrame(filas)
print(R['regla_eleccion']); print('Elegido:',R['nombre'])
print(comparacion.round(5).to_string(index=False))
print('Particiones:',R['particiones'])
px.bar(comparacion,x='modelo',y='rmse',color='particion',barmode='group',
       title='Clases 11 y 12 · RMSE de entrenamiento, validación y prueba',
       labels={'rmse':'RMSE (MW/MW)'}).show()


## Por qué este modelo

La comparación debe considerar error, sobreajuste, interpretación y costo. La regresión lineal da una referencia sencilla; el árbol permite reglas; Random Forest promedia árboles independientes; XGBoost construye correcciones secuenciales. Un R² alto no significa «accuracy»: son métricas de tareas distintas.

In [ ]:
nube=pd.DataFrame(R['nube'])
px.scatter(nube,x='pred',y='real',color='planta',opacity=.5,
           labels={'pred':'Predicción (MW/MW)','real':'Observación (MW/MW)'},
           title='Clase 12 · Predicción vs observación de prueba (muestra visual)').show()
print('Métricas por planta, sobre todos sus registros de prueba:')
print(pd.DataFrame(R['por_planta']).T.round(5))
print('CV de calibración (5 folds por mes, solo train):'); print(pd.DataFrame(R['cv']).round(5))


## Argentina: una evaluación distinta

Se compara regresión múltiple, árbol y RF. Train hasta 2023, selección en 2024 y prueba retrospectiva 2025–septiembre de 2026. Target: factor de capacidad. No se mezclan estas métricas con las de la flota china.

In [ ]:
AR=evidencia['regresion_argentina']
print(AR['nombre'],AR['protocolo'])
print(pd.DataFrame([{'modelo':r['nombre'],**r['test']} for r in AR['comparacion']]).round(5))
print(AR['limite'])
